# AI-Pipeline — Detección y análisis de fuego activo en Colombia (VIIRS + MODIS)

Pipeline refactorizado y corregido a partir del notebook original (`astra-incendios-ia/Untitled0.ipynb`).

**Mejoras principales sobre el original:**
1. **Datos históricos descargados por API** (fuentes SP de FIRMS), no subidos a mano.
2. **Parsing robusto**: detección de delimitador, coordenadas validadas contra el bbox de Colombia (nunca `0.0` a la fuerza).
3. **Objetivos honestos, sin fuga de datos**: el campo que define la etiqueta queda **excluido de las features**.
4. **Cross-sensor validación real**: entrenar en VIIRS SNPP → evaluar en VIIRS NOAA-20 (JPSS1) y MODIS.
5. Desbalance atendido (`class_weight='balanced'`, split estratificado, métricas por clase).

Funciona igual en Colab (clonando el repo) y localmente.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib requests tqdm

import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src import config, clean, download, evaluate, features, load, model, targets
import pandas as pd
import matplotlib.pyplot as plt

print("ROOT:", ROOT)
print("MAP_KEY presente:", bool(config.FIRMS_MAP_KEY))

## 1. Datos históricos (una sola vez)

Descarga por API de FIRMS (fuentes SP = archivo histórico) para el bbox de Colombia. Es reanudable: lo que ya está en `data/raw/` no se vuelve a descargar.
Períodos equivalentes a los del notebook original: VIIRS SNPP 2025, VIIRS NOAA-20 (JPSS1) 2024, MODIS 2025.

In [ ]:
import datetime as dt

PLAN = {
    "viirs_snpp":   ("VIIRS_SNPP_SP",   dt.date(2025, 1, 1), dt.date(2025, 12, 31)),
    "viirs_noaa20": ("VIIRS_NOAA20_SP", dt.date(2024, 1, 1), dt.date(2024, 12, 31)),
    "modis":        ("MODIS_SP",        dt.date(2025, 1, 1), dt.date(2025, 12, 31)),
}

if not config.FIRMS_MAP_KEY:
    # En Colab: from google.colab import userdata; config.FIRMS_MAP_KEY = userdata.get('FIRMS_MAP_KEY')
    raise RuntimeError("Configura FIRMS_MAP_KEY (archivo .env o userdata de Colab)")

for sensor, (source_id, d1, d2) in PLAN.items():
    print(f"== {sensor}: {d1} → {d2}")
    download.descargar_sensor(sensor, source_id, d1, d2, config.RAW_DIR / sensor)
print("\nListo: data/raw/")

## 2. Limpieza y homogenización
- Delimitador detectado con `csv.Sniffer`
- `confidence`: VIIRS (l/n/h) y MODIS (0–100) → `baja / nominal / alta` + puntaje
- Coordenadas: validadas contra bbox de Colombia (se reportan los descartes)

In [ ]:
datasets = {"viirs_snpp": None, "viirs_noaa20": None, "modis": None}

for sensor in PLAN:
    raw = load.load_sensor_dir(config.RAW_DIR / sensor)
    if raw is None:
        print(f"{sensor}: sin datos"); continue
    df, rep = clean.clean_dataset(raw)
    df = features.add_feature_columns(df)
    df = targets.add_targets(df)
    datasets[sensor] = df
    print(sensor, f"{rep['entrada']} → {rep['salida']} registros",
          f"| fuera de bbox: {rep['fuera_bbox']}", f"| duplicados: {rep['duplicados_quitados']}")

unificado = pd.concat([v for v in datasets.values() if v is not None], ignore_index=True)

In [ ]:
print("\nRegistros por sensor:")
print(unificado["instrument"].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=110)
unificado["frp"].clip(upper=100).hist(bins=60, ax=axes[0], color="#d95f02")
axes[0].set_title("FRP (MW, truncado a 100)")
axes[0].set_xlabel("FRP")
unificado["brightness_k"].hist(bins=60, ax=axes[1], color="#7570b3")
axes[1].set_title("Temperatura de brillo 4 µm (K)")
plt.tight_layout(); plt.show()

## 3. Modelos (sin fuga de datos)
Se entrena con VIIRS SNPP y se valida cruzando contra NOAA-20 y MODIS.

Recuerda: al usar `nivel_impacto` se **excluyen** `frp` y `brightness_k` de las features (definen la etiqueta); al usar `confianza_deteccion` se excluyen las columnas de confianza.

In [ ]:
def run_target(target: str, df_train, datasets: dict):
    X = features.make_X(df_train, target)
    y = df_train[target].dropna()
    X = X.loc[y.index]
    res = model.entrenar(X, y)
    print(f"Clases: {sorted(y.unique())}")
    print(f"CV accuracy: {res['cv_accuracy']:.3f} | CV f1_weighted: {res['cv_f1_weighted']:.3f}")
    print(f"Baseline (mayoritaria) accuracy: {res['baseline_acc_test']:.3f}")
    print(evaluate.reporte(res['y_test'], res['modelo'].predict(res['X_test'])))
    return res

res_impacto = run_target('nivel_impacto', datasets['viirs_snpp'], datasets)

In [ ]:
df_n20 = datasets['viirs_noaa20']
X_n20 = features.make_X(df_n20, 'nivel_impacto')
y_n20 = df_n20['nivel_impacto'].dropna().loc[X_n20.index]
X_n20 = model.alinear_features(X_n20.loc[y_n20.index], list(features.make_X(datasets['viirs_snpp'], 'nivel_impacto').columns))
print('Validación cruzada en VIIRS NOAA-20 2024 (n=%d):' % len(y_n20))
print(evaluate.reporte(y_n20, res_impacto['modelo'].predict(X_n20)))

In [ ]:
res_conf = run_target('confianza_deteccion', datasets['viirs_snpp'], datasets)

## Conclusiones del demo
- El modelo es honesto: sus métricas reflectan la información que realmente hay en posición + estación + horario.
- Sirve como baseline sólido para comparar cuando se crucen covariables externas (cobertura terrestre ESA WorldCover, caminos, población, precipitación).
- La explicación completa del pipeline y sus resultados está en `README.md` y `outputs/reporte_modelo.md`.